# Geofabrik Data Download

In [0]:
%sh
wget -O /Volumes/prd_mega/saiana95/vaiana95/OSM/geofabrik/antartica.osm.pbf https://download.geofabrik.de/antarctica-latest.osm.pbf

In [0]:
%sh
set -e

DEST_DIR="/Volumes/prd_mega/saiana95/vaiana95/OSM/geofabrik"

declare -A FILES=(
  [africa]="africa-latest.osm.pbf"
  [antarctica]="antartica.osm.pbf"          # already downloaded — kept here so re-runs verify/skip it too
  [asia]="asia-latest.osm.pbf"
  [australia-oceania]="australia-oceania-latest.osm.pbf"
  [central-america]="central-america-latest.osm.pbf"
  [europe]="europe-latest.osm.pbf"
  [north-america]="north-america-latest.osm.pbf"
  [south-america]="south-america-latest.osm.pbf"
)

for region in "${!FILES[@]}"; do
  fname="${FILES[$region]}"
  dest="${DEST_DIR}/${fname}"
  url="https://download.geofabrik.de/${region}-latest.osm.pbf"

  if [ -f "$dest" ] && [ -s "$dest" ]; then
    echo "⏭  ${region}: already present (${dest}) — skipping"
    continue
  fi

  echo "⬇️  ${region}: downloading ${url}"
  wget -c --tries=5 --timeout=60 --waitretry=15 \
       -O "$dest" "$url"

  if [ -s "$dest" ]; then
    echo "✅ ${region}: done ($(du -h "$dest" | cut -f1))"
  else
    echo "❌ ${region}: download produced an empty file"
    rm -f "$dest"
  fi
done

echo "----------------------------------------"
echo "Final contents of ${DEST_DIR}:"
ls -lh "$DEST_DIR"

# Download the staged data as parts

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, MapType, StructType, BinaryType

output_dir = "/Volumes/prd_mega/sgpbpi163/vgpbpi163/OSM/extraction"
tmp_dir    = f"{output_dir}/_tmp_road_networks_27_09"
final_path = f"{output_dir}/road_networks_27_09.csv"

df = spark.table("prd_mega.sgpbpi163.osm_road_staging")

# CSV can't store arrays/maps/structs/binary (common in OSM tags/geometry) -> convert to strings
for f in df.schema.fields:
    if isinstance(f.dataType, (ArrayType, MapType, StructType)):
        df = df.withColumn(f.name, F.to_json(F.col(f.name)))
    elif isinstance(f.dataType, BinaryType):
        df = df.withColumn(f.name, F.base64(F.col(f.name)))

(df.repartition(1)
   .write.mode("overwrite")
   .option("header", True)
   .option("quote", '"').option("escape", '"')
   .option("maxRecordsPerFile", 1_000_000)   # each file stays under Excel's row limit
   .csv(tmp_dir))

parts = sorted(f.path for f in dbutils.fs.ls(tmp_dir)
               if f.name.startswith("part-") and f.name.endswith(".csv"))
for i, p in enumerate(parts, 1):
    dbutils.fs.mv(p, f"{output_dir}/road_networks_27_09_part{i:02d}.csv")
dbutils.fs.rm(tmp_dir, recurse=True)

print(f"Saved {len(parts)} files to {output_dir}")

# Download the staged data as country-wise

In [0]:
import re
from urllib.parse import unquote
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, MapType, StructType, BinaryType

output_dir = "/Volumes/prd_mega/sgpbpi163/vgpbpi163/OSM/extraction"
tmp_dir    = f"{output_dir}/_tmp_road_networks_27_09"

df = spark.table("prd_mega.sgpbpi163.osm_road_staging")

# CSV can't store arrays/maps/structs/binary -> convert to strings
for f in df.schema.fields:
    if isinstance(f.dataType, (ArrayType, MapType, StructType)):
        df = df.withColumn(f.name, F.to_json(F.col(f.name)))
    elif isinstance(f.dataType, BinaryType):
        df = df.withColumn(f.name, F.base64(F.col(f.name)))

# Split on a copy so country_name itself stays inside each CSV
df = df.withColumn("_split_country", F.col("country_name"))

(df.repartition("_split_country")      # each country handled by one task -> one file per country
   .write.mode("overwrite")
   .partitionBy("_split_country")
   .option("header", True)
   .option("quote", '"').option("escape", '"')
   .csv(tmp_dir))

# Move each country's part file to road_networks_27_09_{country_name}.csv
for folder in dbutils.fs.ls(tmp_dir):
    if not folder.name.startswith("_split_country="):
        continue
    country = unquote(folder.name.rstrip("/").split("=", 1)[1])
    if country == "__HIVE_DEFAULT_PARTITION__":      # rows where country_name is null
        country = "unknown"
    safe_name = re.sub(r"[^\w\-]+", "_", country).strip("_")
    part = [f.path for f in dbutils.fs.ls(folder.path)
            if f.name.startswith("part-") and f.name.endswith(".csv")][0]
    dest = f"/Volumes/prd_mega/sgpbpi163/vgpbpi163/OSM/extraction/osm_country_wise_data/road_networks_27_09_{safe_name}.csv"
    dbutils.fs.rm(dest)                               # allows re-running without clashes
    dbutils.fs.mv(part, dest)

dbutils.fs.rm(tmp_dir, recurse=True)
print(f"Saved per-country CSVs to {output_dir}")